# Mini-Projeto 03 — Biblioteca escolar
**Gabriel Rafá Martins Freire — matrícula 20230145310**  
Modalidade individual. Domínio novo, independente dos projetos de Sekiro.

**Objetivo:** representar pessoas, livros, locais e empréstimos em RDF/RDFS/OWL, salvar a base em Turtle e consultá-la com RDFLib e SPARQL.

Execute todas as células de código de cima para baixo. Este notebook inclui a mesma base `.ttl` e o mesmo código de consultas da aplicação, para funcionar sozinho no Colab. A entrega principal também possui `biblioteca.ttl`, `consultas.py` e `README.md`. Os resultados abaixo foram obtidos em execução local real.

Todos os nomes, títulos e registros da base são **fictícios**, destinados à demonstração. O autor acadêmico indicado acima é a identificação da entrega, não uma pessoa do cadastro fictício.

## 1. A ideia em linguagem simples
Uma biblioteca precisa saber quais livros possui, quem os escreveu, onde ficam e quem está com um exemplar.

Uma **tripla RDF** é uma informação com três partes:

| Sujeito | Predicado | Objeto |
|---|---|---|
| livro01 | escritoPor | autor1 |
| livro01 | paginas | 210 |
| emprestimo01 | emprestadoPara | ana |

`ex:livro01` identifica um recurso. `ex:` é uma abreviação do endereço `https://example.org/biblioteca#`. O endereço funciona como identificador e não precisa ser uma página acessível. `a` é abreviação Turtle para `rdf:type`.

- **RDF:** representa os fatos como triplas.
- **RDFS:** acrescenta classes, subclasses, domínio e alcance das propriedades.
- **OWL:** permite expressar inversão, transitividade, simetria, funcionalidade e disjunção.
- **Turtle:** formato de texto usado para escrever a base.
- **SPARQL:** linguagem de consultas e atualizações dessa base.

Cada recurso Livro representa **um exemplar físico**, com código de inventário próprio. `guardadoEm` representa a estante habitual, não a localização física atual de um livro emprestado. Cada empréstimo trata de um exemplar. A data de referência das consultas é fixa: **03/11/2026**.

## 2. Instalação
Use Python 3.12 para reproduzir o ambiente validado. No Colab, execute esta célula primeiro. Internet é necessária apenas para instalar os pacotes; a base é local e não consulta serviços externos.

In [ ]:
%pip install --quiet rdflib==7.1.4 owlrl==7.1.4 pyparsing==3.3.3

Note: you may need to restart the kernel to use updated packages.


## 3. Importações e prefixos
`Graph` armazena a base; `Namespace` evita repetir endereços. `owlrl` será usado somente na demonstração adicional de inferência. RDFLib não aplica automaticamente todas as construções OWL ao carregar Turtle.

In [ ]:
from pathlib import Path
from datetime import date
from rdflib import Graph, Namespace, RDF, RDFS, OWL, XSD, Literal, URIRef
from owlrl import DeductiveClosure, OWLRL_Semantics

EX = Namespace('https://example.org/biblioteca#')
BASE = Path('biblioteca.ttl')
PREFIXOS = '''PREFIX ex: <https://example.org/biblioteca#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
'''

## 4. Classes e taxonomia
A base contém 14 classes. A raiz é `Entidade`. Um exemplo com vários níveis abaixo dela é **Entidade → ItemAcervo → Livro → Romance**.

| Classe | Superclasse direta | Exemplo |
|---|---|---|
| Entidade | — | Raiz geral |
| Pessoa | Entidade | Generalização de alunos, professores e autores |
| Aluno | Pessoa | ana |
| Professor | Pessoa | profLia |
| Autor | Pessoa | autor1 |
| ItemAcervo | Entidade | Generalização dos itens do acervo |
| Livro | ItemAcervo | Tipo geral dos exemplares |
| Romance | Livro | livro01 |
| LivroDidatico | Livro | livro05 |
| LivroInfantil | Livro | livro08 |
| Local | Entidade | Generalização dos locais |
| Biblioteca | Local | central |
| Estante | Local | estanteRomance |
| Emprestimo | Entidade | emprestimo01 |

Uma classe é uma categoria; uma instância é um elemento concreto. `Aluno` é classe, `ana` é instância. `Romance` é classe, `livro01` é instância. A base declara 33 indivíduos: 6 alunos, 2 professores, 4 autores, 10 livros, 2 bibliotecas, 4 estantes e 5 empréstimos.

## 5. Propriedades e cinco construções OWL
Há 10 propriedades de objeto (recurso → recurso) e 9 de dados (recurso → literal). Todas possuem `rdfs:domain` e `rdfs:range`.

Exemplo de objeto: `escritoPor`, domínio Livro e alcance Autor. Exemplo de dado: `paginas`, domínio Livro e alcance `xsd:integer`.

| Construção OWL | Aplicação | Justificativa |
|---|---|---|
| `owl:inverseOf` | autorDe é inversa de escritoPor | Se o livro é escrito por um autor, esse autor é autor do livro. |
| `owl:TransitiveProperty` | localizadoEm | Livro na estante e estante na biblioteca permitem inferir a localização institucional do livro. |
| `owl:SymmetricProperty` | parceiraDe | A parceria entre as duas bibliotecas é recíproca. |
| `owl:FunctionalProperty` | itemEmprestado e codigoInventario | Um registro trata de no máximo um exemplar; cada exemplar tem no máximo um valor de código. |
| `owl:disjointWith` | Livro é disjunto de Pessoa e de Local | Um exemplar não pode ser pessoa nem local no domínio. |

`guardadoEm` e `pertenceBiblioteca` são subpropriedades de `localizadoEm`, permitindo a cadeia de inferência. **Transitividade não significa que o livro emprestado está fisicamente na biblioteca:** usamos a localização habitual no acervo.

**Cuidado:** domain/range inferem tipos; não rejeitam entradas como um formulário. Funcionalidade significa no máximo um valor por sujeito, não unicidade global nem obrigatoriedade. Para propriedades de objeto, dois valores podem levar a uma conclusão de igualdade. A ausência de um fato não prova sua falsidade em OWL (mundo aberto). As assertivas Python verificam regras de qualidade do cadastro separadamente.

## 6. Base Turtle completa
A string abaixo é uma cópia exata do arquivo `biblioteca.ttl` do pacote. Ela torna o notebook independente de upload de arquivos auxiliares. O parser lê essa mesma sintaxe Turtle. Na aplicação `.py`, a leitura é feita diretamente do arquivo.

As propriedades `ativo` e as datas descrevem o estado do cadastro, não são atualizadas pelo relógio. Não existe integração com biblioteca real.

In [ ]:
BASE_TTL = r"""@prefix ex: <https://example.org/biblioteca#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

# Dados inteiramente fictícios, destinados à demonstração acadêmica.
# Cada Livro representa um exemplar físico com código de inventário.
# guardadoEm indica a estante habitual, mesmo quando o livro está emprestado.
ex:OntologiaBiblioteca a owl:Ontology ;
    rdfs:label "Biblioteca escolar didática"@pt ;
    rdfs:comment "Mini-Projeto 03. Gabriel Rafá Martins Freire, 20230145310. Cenário fictício; datas de referência fixas."@pt .

# 1. CLASSES: taxonomia com vários níveis abaixo de Entidade.
ex:Entidade a owl:Class ; rdfs:label "Entidade"@pt .
ex:Pessoa a owl:Class ; rdfs:label "Pessoa"@pt ; rdfs:subClassOf ex:Entidade .
ex:Aluno a owl:Class ; rdfs:label "Aluno"@pt ; rdfs:subClassOf ex:Pessoa .
ex:Professor a owl:Class ; rdfs:label "Professor"@pt ; rdfs:subClassOf ex:Pessoa .
ex:Autor a owl:Class ; rdfs:label "Autor"@pt ; rdfs:subClassOf ex:Pessoa .
ex:ItemAcervo a owl:Class ; rdfs:label "Item do acervo"@pt ; rdfs:subClassOf ex:Entidade .
ex:Livro a owl:Class ; rdfs:label "Livro"@pt ; rdfs:subClassOf ex:ItemAcervo .
ex:Romance a owl:Class ; rdfs:label "Romance"@pt ; rdfs:subClassOf ex:Livro .
ex:LivroDidatico a owl:Class ; rdfs:label "Livro didático"@pt ; rdfs:subClassOf ex:Livro .
ex:LivroInfantil a owl:Class ; rdfs:label "Livro infantil"@pt ; rdfs:subClassOf ex:Livro .
ex:Local a owl:Class ; rdfs:label "Local"@pt ; rdfs:subClassOf ex:Entidade .
ex:Biblioteca a owl:Class ; rdfs:label "Biblioteca"@pt ; rdfs:subClassOf ex:Local .
ex:Estante a owl:Class ; rdfs:label "Estante"@pt ; rdfs:subClassOf ex:Local .
ex:Emprestimo a owl:Class ; rdfs:label "Empréstimo"@pt ; rdfs:subClassOf ex:Entidade .

# Disjunção: um livro não pode ser uma pessoa ou um local neste domínio.
ex:Livro owl:disjointWith ex:Pessoa, ex:Local .

# 2. PROPRIEDADES DE OBJETO: ligam dois recursos.
ex:escritoPor a owl:ObjectProperty ; rdfs:domain ex:Livro ; rdfs:range ex:Autor ; rdfs:label "Autor do exemplar"@pt  .
ex:autorDe a owl:ObjectProperty ; rdfs:domain ex:Autor ; rdfs:range ex:Livro ; rdfs:label "Livros do autor"@pt ; owl:inverseOf ex:escritoPor .
ex:localizadoEm a owl:ObjectProperty, owl:TransitiveProperty ; rdfs:domain ex:Entidade ; rdfs:range ex:Local ; rdfs:label "Localização direta ou indireta"@pt  .
ex:guardadoEm a owl:ObjectProperty ; rdfs:domain ex:Livro ; rdfs:range ex:Estante ; rdfs:label "Estante de guarda habitual"@pt ; rdfs:subPropertyOf ex:localizadoEm .
ex:pertenceBiblioteca a owl:ObjectProperty ; rdfs:domain ex:Estante ; rdfs:range ex:Biblioteca ; rdfs:label "Biblioteca da estante"@pt ; rdfs:subPropertyOf ex:localizadoEm .
ex:emprestadoPara a owl:ObjectProperty ; rdfs:domain ex:Emprestimo ; rdfs:range ex:Pessoa ; rdfs:label "Pessoa responsável pelo empréstimo"@pt  .
ex:itemEmprestado a owl:ObjectProperty, owl:FunctionalProperty ; rdfs:domain ex:Emprestimo ; rdfs:range ex:Livro ; rdfs:label "Único exemplar de cada registro"@pt  .
ex:registradoEm a owl:ObjectProperty ; rdfs:domain ex:Emprestimo ; rdfs:range ex:Biblioteca ; rdfs:label "Biblioteca que registra o empréstimo"@pt  .
ex:parceiraDe a owl:ObjectProperty, owl:SymmetricProperty ; rdfs:domain ex:Biblioteca ; rdfs:range ex:Biblioteca ; rdfs:label "Parceria recíproca entre bibliotecas"@pt  .
ex:estaCom a owl:ObjectProperty ; rdfs:domain ex:Pessoa ; rdfs:range ex:Livro ; rdfs:label "Relação derivada para consulta de empréstimos ativos"@pt  .

# 3. PROPRIEDADES DE DADOS: ligam um recurso a um literal tipado.
ex:nome a owl:DatatypeProperty ; rdfs:domain ex:Entidade ; rdfs:range xsd:string ; rdfs:label "Nome de pessoa ou local"@pt .
ex:titulo a owl:DatatypeProperty ; rdfs:domain ex:Livro ; rdfs:range xsd:string ; rdfs:label "Título do exemplar"@pt .
ex:anoPublicacao a owl:DatatypeProperty ; rdfs:domain ex:Livro ; rdfs:range xsd:integer ; rdfs:label "Ano da publicação"@pt .
ex:paginas a owl:DatatypeProperty ; rdfs:domain ex:Livro ; rdfs:range xsd:integer ; rdfs:label "Número de páginas"@pt .
ex:codigoInventario a owl:DatatypeProperty, owl:FunctionalProperty ; rdfs:domain ex:Livro ; rdfs:range xsd:string ; rdfs:label "Código único atribuído a cada exemplar"@pt .
ex:matricula a owl:DatatypeProperty ; rdfs:domain ex:Aluno ; rdfs:range xsd:string ; rdfs:label "Matrícula fictícia do aluno"@pt .
ex:dataEmprestimo a owl:DatatypeProperty ; rdfs:domain ex:Emprestimo ; rdfs:range xsd:date ; rdfs:label "Data de retirada"@pt .
ex:dataPrevista a owl:DatatypeProperty ; rdfs:domain ex:Emprestimo ; rdfs:range xsd:date ; rdfs:label "Data prevista de devolução"@pt .
ex:ativo a owl:DatatypeProperty ; rdfs:domain ex:Emprestimo ; rdfs:range xsd:boolean ; rdfs:label "Registro ainda não encerrado"@pt .

# 4. PESSOAS: 6 alunos, 2 professores e 4 autores fictícios.
ex:ana a owl:NamedIndividual, ex:Aluno ; ex:nome "Ana Lima"^^xsd:string ; ex:matricula "A001"^^xsd:string .
ex:bruno a owl:NamedIndividual, ex:Aluno ; ex:nome "Bruno Costa"^^xsd:string ; ex:matricula "A002"^^xsd:string .
ex:carla a owl:NamedIndividual, ex:Aluno ; ex:nome "Carla Alves"^^xsd:string ; ex:matricula "A003"^^xsd:string .
ex:diego a owl:NamedIndividual, ex:Aluno ; ex:nome "Diego Rocha"^^xsd:string ; ex:matricula "A004"^^xsd:string .
ex:elisa a owl:NamedIndividual, ex:Aluno ; ex:nome "Elisa Melo"^^xsd:string ; ex:matricula "A005"^^xsd:string .
ex:felipe a owl:NamedIndividual, ex:Aluno ; ex:nome "Felipe Silva"^^xsd:string ; ex:matricula "A006"^^xsd:string .
ex:profLia a owl:NamedIndividual, ex:Professor ; ex:nome "Lia Santos"^^xsd:string .
ex:profRui a owl:NamedIndividual, ex:Professor ; ex:nome "Rui Gomes"^^xsd:string .
ex:autor1 a owl:NamedIndividual, ex:Autor ; ex:nome "Clara Nunes de Souza"^^xsd:string .
ex:autor2 a owl:NamedIndividual, ex:Autor ; ex:nome "Miguel Torres Lima"^^xsd:string .
ex:autor3 a owl:NamedIndividual, ex:Autor ; ex:nome "Helena Martins Reis"^^xsd:string .
ex:autor4 a owl:NamedIndividual, ex:Autor ; ex:nome "Paulo Azevedo Dias"^^xsd:string .

# 5. LOCAIS: 2 bibliotecas e 4 estantes.
ex:central a owl:NamedIndividual, ex:Biblioteca ; ex:nome "Biblioteca Central da Escola"^^xsd:string .
ex:anexo a owl:NamedIndividual, ex:Biblioteca ; ex:nome "Biblioteca do Anexo"^^xsd:string .
ex:central ex:parceiraDe ex:anexo .
ex:estanteRomance a owl:NamedIndividual, ex:Estante ; ex:nome "Estante de romances"^^xsd:string ; ex:pertenceBiblioteca ex:central .
ex:estanteDidatica a owl:NamedIndividual, ex:Estante ; ex:nome "Estante de livros didáticos"^^xsd:string ; ex:pertenceBiblioteca ex:central .
ex:estanteInfantil a owl:NamedIndividual, ex:Estante ; ex:nome "Estante infantil"^^xsd:string ; ex:pertenceBiblioteca ex:central .
ex:estanteAnexo a owl:NamedIndividual, ex:Estante ; ex:nome "Estante do anexo"^^xsd:string ; ex:pertenceBiblioteca ex:anexo .

# 6. LIVROS: 10 exemplares fictícios, sem ISBNs ou obras reais atribuídas.
ex:livro01 a owl:NamedIndividual, ex:Romance ;
    ex:titulo "O Caminho da Serra"^^xsd:string ; ex:anoPublicacao "2020"^^xsd:integer ;
    ex:paginas "210"^^xsd:integer ; ex:codigoInventario "L001"^^xsd:string ;
    ex:escritoPor ex:autor1 ; ex:guardadoEm ex:estanteRomance .
ex:livro02 a owl:NamedIndividual, ex:Romance ;
    ex:titulo "Cartas do Amanhã"^^xsd:string ; ex:anoPublicacao "2021"^^xsd:integer ;
    ex:paginas "180"^^xsd:integer ; ex:codigoInventario "L002"^^xsd:string ;
    ex:escritoPor ex:autor2 ; ex:guardadoEm ex:estanteRomance .
ex:livro03 a owl:NamedIndividual, ex:Romance ;
    ex:titulo "O Mistério da Ponte"^^xsd:string ; ex:anoPublicacao "2022"^^xsd:integer ;
    ex:paginas "240"^^xsd:integer ; ex:codigoInventario "L003"^^xsd:string ;
    ex:escritoPor ex:autor1 ; ex:guardadoEm ex:estanteRomance .
ex:livro04 a owl:NamedIndividual, ex:Romance ;
    ex:titulo "A Casa do Lago"^^xsd:string ; ex:anoPublicacao "2019"^^xsd:integer ;
    ex:paginas "160"^^xsd:integer ; ex:codigoInventario "L004"^^xsd:string ;
    ex:escritoPor ex:autor3 ; ex:guardadoEm ex:estanteAnexo .
ex:livro05 a owl:NamedIndividual, ex:LivroDidatico ;
    ex:titulo "Matemática em Passos"^^xsd:string ; ex:anoPublicacao "2023"^^xsd:integer ;
    ex:paginas "120"^^xsd:integer ; ex:codigoInventario "L005"^^xsd:string ;
    ex:escritoPor ex:autor4 ; ex:guardadoEm ex:estanteDidatica .
ex:livro06 a owl:NamedIndividual, ex:LivroDidatico ;
    ex:titulo "Ciência na Escola"^^xsd:string ; ex:anoPublicacao "2024"^^xsd:integer ;
    ex:paginas "150"^^xsd:integer ; ex:codigoInventario "L006"^^xsd:string ;
    ex:escritoPor ex:autor4 ; ex:guardadoEm ex:estanteDidatica .
ex:livro07 a owl:NamedIndividual, ex:LivroDidatico ;
    ex:titulo "História em Mapas"^^xsd:string ; ex:anoPublicacao "2022"^^xsd:integer ;
    ex:paginas "200"^^xsd:integer ; ex:codigoInventario "L007"^^xsd:string ;
    ex:escritoPor ex:autor4 ; ex:guardadoEm ex:estanteAnexo .
ex:livro08 a owl:NamedIndividual, ex:LivroInfantil ;
    ex:titulo "A Estrela Curiosa"^^xsd:string ; ex:anoPublicacao "2021"^^xsd:integer ;
    ex:paginas "48"^^xsd:integer ; ex:codigoInventario "L008"^^xsd:string ;
    ex:escritoPor ex:autor3 ; ex:guardadoEm ex:estanteInfantil .
ex:livro09 a owl:NamedIndividual, ex:LivroInfantil ;
    ex:titulo "O Jardim das Cores"^^xsd:string ; ex:anoPublicacao "2023"^^xsd:integer ;
    ex:paginas "64"^^xsd:integer ; ex:codigoInventario "L009"^^xsd:string ;
    ex:escritoPor ex:autor2 ; ex:guardadoEm ex:estanteInfantil .
ex:livro10 a owl:NamedIndividual, ex:LivroInfantil ;
    ex:titulo "A Viagem da Semente"^^xsd:string ; ex:anoPublicacao "2024"^^xsd:integer ;
    ex:paginas "80"^^xsd:integer ; ex:codigoInventario "L010"^^xsd:string ;
    ex:escritoPor ex:autor1 ; ex:guardadoEm ex:estanteInfantil .

# 7. EMPRÉSTIMOS: um registro por exemplar; todos ativos no cenário inicial.
ex:emprestimo01 a owl:NamedIndividual, ex:Emprestimo ;
    ex:emprestadoPara ex:ana ; ex:itemEmprestado ex:livro01 ; ex:registradoEm ex:central ;
    ex:dataEmprestimo "2026-11-03"^^xsd:date ; ex:dataPrevista "2026-11-17"^^xsd:date ; ex:ativo "true"^^xsd:boolean .
ex:emprestimo02 a owl:NamedIndividual, ex:Emprestimo ;
    ex:emprestadoPara ex:bruno ; ex:itemEmprestado ex:livro05 ; ex:registradoEm ex:central ;
    ex:dataEmprestimo "2026-10-16"^^xsd:date ; ex:dataPrevista "2026-10-30"^^xsd:date ; ex:ativo "true"^^xsd:boolean .
ex:emprestimo03 a owl:NamedIndividual, ex:Emprestimo ;
    ex:emprestadoPara ex:carla ; ex:itemEmprestado ex:livro08 ; ex:registradoEm ex:central ;
    ex:dataEmprestimo "2026-10-21"^^xsd:date ; ex:dataPrevista "2026-11-04"^^xsd:date ; ex:ativo "true"^^xsd:boolean .
ex:emprestimo04 a owl:NamedIndividual, ex:Emprestimo ;
    ex:emprestadoPara ex:profLia ; ex:itemEmprestado ex:livro07 ; ex:registradoEm ex:anexo ;
    ex:dataEmprestimo "2026-10-14"^^xsd:date ; ex:dataPrevista "2026-10-28"^^xsd:date ; ex:ativo "true"^^xsd:boolean .
ex:emprestimo05 a owl:NamedIndividual, ex:Emprestimo ;
    ex:emprestadoPara ex:diego ; ex:itemEmprestado ex:livro03 ; ex:registradoEm ex:central ;
    ex:dataEmprestimo "2026-10-27"^^xsd:date ; ex:dataPrevista "2026-11-10"^^xsd:date ; ex:ativo "true"^^xsd:boolean .
"""
g = Graph().parse(data=BASE_TTL, format="turtle")
print("Turtle carregado:", len(g), "triplas.")

Turtle carregado: 314 triplas.


## 7. Funções auxiliares e conferência dos requisitos
As funções abaixo apenas copiam o grafo, formatam a saída e verificam a qualidade dos dados. A quantidade de indivíduos é contada pelas declarações `owl:NamedIndividual`; classes e propriedades não são contadas como pessoas ou livros. As contagens são feitas **antes** de aplicar inferência.

In [ ]:
def copiar(g):
    novo = Graph()
    for prefixo, uri in g.namespaces():
        novo.bind(prefixo, uri)
    for tripla in g:
        novo.add(tripla)
    return novo


def curto(g, termo):
    if isinstance(termo, Literal):
        return termo.n3(g.namespace_manager)
    return g.namespace_manager.normalizeUri(termo)


def mostrar_triplas(g, triplas):
    linhas = sorted(tuple(curto(g, t) for t in tripla) for tripla in triplas)
    for linha in linhas:
        print(' | '.join(linha))
    print(f'Total: {len(linhas)} triplas')
    return linhas


def validar_base(g):
    """Verificações do conjunto de dados; não são restrições impostas por OWL."""
    classes = set(g.subjects(RDF.type, OWL.Class))
    objetos = set(g.subjects(RDF.type, OWL.ObjectProperty))
    dados = set(g.subjects(RDF.type, OWL.DatatypeProperty))
    individuos = set(g.subjects(RDF.type, OWL.NamedIndividual))
    assert len(classes) == 14 and len(objetos) == 10 and len(dados) == 9
    assert len(individuos) == 33 and len(g) >= 50
    assert (EX.Livro, RDFS.subClassOf, EX.ItemAcervo) in g
    assert (EX.ItemAcervo, RDFS.subClassOf, EX.Entidade) in g
    for prop in objetos | dados:
        assert len(list(g.objects(prop, RDFS.domain))) == 1
        assert len(list(g.objects(prop, RDFS.range))) == 1
    for pessoa in g.subjects(RDF.type, EX.Aluno):
        assert (pessoa, EX.matricula, None) in g
    codigos = list(g.objects(None, EX.codigoInventario))
    assert len(codigos) == len(set(codigos)) == 10
    for livro in g.subjects(EX.titulo, None):
        assert len(list(g.objects(livro, EX.codigoInventario))) == 1
        assert int(g.value(livro, EX.paginas)) > 0
    emprestados = []
    for emp in g.subjects(RDF.type, EX.Emprestimo):
        assert len(list(g.objects(emp, EX.itemEmprestado))) == 1
        assert g.value(emp, EX.dataEmprestimo).toPython() <= g.value(emp, EX.dataPrevista).toPython()
        if g.value(emp, EX.ativo).toPython():
            emprestados.append(g.value(emp, EX.itemEmprestado))
    assert len(emprestados) == len(set(emprestados))
    for pred in [EX.nome, EX.titulo, EX.codigoInventario, EX.matricula]:
        assert all(v.datatype == XSD.string for v in g.objects(None, pred))
    print(f'BASE: {len(classes)} classes; {len(objetos)} propriedades de objeto; '
          f'{len(dados)} de dados; {len(individuos)} indivíduos; {len(g)} triplas.')
    return dict(classes=len(classes), propriedades_objeto=len(objetos),
                propriedades_dados=len(dados), individuos=len(individuos), triplas=len(g))


resumo = validar_base(g)

BASE: 14 classes; 10 propriedades de objeto; 9 de dados; 33 indivíduos; 314 triplas.


## 8. Parte 1: seis consultas com `g.triples()`
O padrão tem a forma `(sujeito, predicado, objeto)`. `None` significa “qualquer valor” naquela posição.

| Consulta | Pergunta | Padrão | Resultado esperado |
|---|---|---|---|
| T1 | Quais os dados do livro01? | (livro01, None, None) | 8 triplas |
| T2 | Quais as relações de autoria? | (None, escritoPor, None) | 10 triplas |
| T3 | Quem aponta para Ana? | (None, None, ana) | 1 tripla |
| T4 | Quais os alunos? | (None, rdf:type, Aluno) | 6 triplas |
| T5 | Qual o autor do livro01? | (livro01, escritoPor, None) | 1 tripla |
| T6 | Quais livros ficam na estante infantil? | (None, guardadoEm, estanteInfantil) | 3 triplas |

Cada consulta está comentada no código e imprime número, propósito e resultado. A quantidade é verificada com `assert`.

In [ ]:
def parte1(g):
    """Seis consultas por padrões. None é um curinga, não um valor do grafo."""
    # T1: sujeito fixo. Esperado: as 8 triplas declaradas do livro01.
    # T2: predicado fixo. Esperado: 10 relações livro -> autor.
    # T3: objeto fixo. Esperado: emprestimo01 -> emprestadoPara -> ana (1).
    # T4: predicado e objeto fixos. Esperado: os 6 alunos declarados.
    # T5: sujeito e predicado fixos. Esperado: autor1, autor do livro01 (1).
    # T6: predicado e objeto fixos. Esperado: livro08, livro09, livro10 (3).
    consultas = [
        ('T1', 'Dados do livro01 (sujeito fixo)', (EX.livro01, None, None), 8),
        ('T2', 'Relações de autoria (predicado fixo)', (None, EX.escritoPor, None), 10),
        ('T3', 'Quem se relaciona com Ana? (objeto fixo)', (None, None, EX.ana), 1),
        ('T4', 'Alunos declarados', (None, RDF.type, EX.Aluno), 6),
        ('T5', 'Autor do livro01', (EX.livro01, EX.escritoPor, None), 1),
        ('T6', 'Livros da estante infantil', (None, EX.guardadoEm, EX.estanteInfantil), 3),
    ]
    for numero, proposito, padrao, esperado in consultas:
        print(f'\n[{numero}] {proposito}. Esperado: {esperado} triplas.')
        resultados = list(g.triples(padrao))  # Consulta obrigatória com g.triples().
        assert len(resultados) == esperado, numero
        mostrar_triplas(g, resultados)


parte1(g)


[T1] Dados do livro01 (sujeito fixo). Esperado: 8 triplas.
ex:livro01 | ex:anoPublicacao | "2020"^^xsd:integer
ex:livro01 | ex:codigoInventario | "L001"^^xsd:string
ex:livro01 | ex:escritoPor | ex:autor1
ex:livro01 | ex:guardadoEm | ex:estanteRomance
ex:livro01 | ex:paginas | "210"^^xsd:integer
ex:livro01 | ex:titulo | "O Caminho da Serra"^^xsd:string
ex:livro01 | rdf:type | ex:Romance
ex:livro01 | rdf:type | owl:NamedIndividual
Total: 8 triplas

[T2] Relações de autoria (predicado fixo). Esperado: 10 triplas.
ex:livro01 | ex:escritoPor | ex:autor1
ex:livro02 | ex:escritoPor | ex:autor2
ex:livro03 | ex:escritoPor | ex:autor1
ex:livro04 | ex:escritoPor | ex:autor3
ex:livro05 | ex:escritoPor | ex:autor4
ex:livro06 | ex:escritoPor | ex:autor4
ex:livro07 | ex:escritoPor | ex:autor4
ex:livro08 | ex:escritoPor | ex:autor3
ex:livro09 | ex:escritoPor | ex:autor2
ex:livro10 | ex:escritoPor | ex:autor1
Total: 10 triplas

[T3] Quem se relaciona com Ana? (objeto fixo). Esperado: 1 triplas.
ex:emp

## 9. Parte 2: oito consultas SPARQL de leitura
| Consulta | Recurso | Pergunta e resultado esperado |
|---|---|---|
| S1 | SELECT + ORDER BY + caminho de classes | Catálogo com autores: 10 linhas |
| S2 | SELECT + FILTER + ORDER BY | Até 120 páginas: 4 livros |
| S3 | SELECT + COUNT + GROUP BY | Contagem por autor: 4 linhas, quantidades 3,3,2,2 |
| S4 | ASK | Ana está com livro01? True |
| S5 | CONSTRUCT | Novo grafo pessoa → estaCom → livro: 5 triplas |
| S6 | SELECT + FILTER de data | Vencidos antes de 03/11/2026: 2 empréstimos |
| S7 | SELECT + caminho de propriedades | Acervo habitual da central: 8 livros |
| S8 | SELECT + FILTER NOT EXISTS | Sem empréstimo ativo registrado: 5 livros |

`SELECT` produz linhas; `ASK` responde verdadeiro/falso; `CONSTRUCT` produz um novo grafo e **não altera a base original**. `ORDER BY` ordena; `FILTER` limita os resultados; `COUNT` conta.

Em S1, `rdf:type/rdfs:subClassOf*` percorre a taxonomia sem executar o raciocinador. S8 significa “não existe esse registro no grafo consultado”; não é prova de disponibilidade no mundo real nem negação lógica em OWL.

In [ ]:
# Oito consultas SPARQL de leitura. Executadas na base explícita, sem inferência.
LEITURAS = [
    ('S1', 'Catálogo com autores, ordenado por título', '''
SELECT DISTINCT ?titulo ?autor WHERE {
  ?livro rdf:type/rdfs:subClassOf* ex:Livro ; ex:titulo ?titulo ; ex:escritoPor ?a .
  ?a ex:nome ?autor .
} ORDER BY ?titulo
''', 10),  # Esperado: 10 linhas. O caminho inclui subclasses de Livro.
    ('S2', 'Leituras curtas: até 120 páginas', '''
SELECT ?titulo ?paginas WHERE {
  ?livro ex:titulo ?titulo ; ex:paginas ?paginas .
  FILTER(?paginas <= 120)
} ORDER BY ?paginas ?titulo
''', 4),  # Esperado: 48, 64, 80 e 120 páginas.
    ('S3', 'Quantidade de exemplares por autor', '''
SELECT ?autor (COUNT(?livro) AS ?quantidade) WHERE {
  ?livro ex:escritoPor ?a . ?a ex:nome ?autor .
} GROUP BY ?autor ORDER BY DESC(?quantidade) ?autor
''', 4),  # Esperado: Clara 3, Paulo 3, Helena 2 e Miguel 2.
    ('S4', 'Ana está com o livro01 em um empréstimo ativo?', '''
ASK { ?emp a ex:Emprestimo ; ex:emprestadoPara ex:ana ;
      ex:itemEmprestado ex:livro01 ; ex:ativo true . }
''', True),  # Esperado: True, por emprestimo01.
    ('S5', 'Construir um grafo pessoa -> estaCom -> livro', '''
CONSTRUCT { ?pessoa ex:estaCom ?livro . }
WHERE { ?emp a ex:Emprestimo ; ex:emprestadoPara ?pessoa ;
        ex:itemEmprestado ?livro ; ex:ativo true . }
''', 5),  # Esperado: grafo separado com 5 triplas; não altera g.
    ('S6', 'Empréstimos vencidos na data fixa 03/11/2026', '''
SELECT ?emp ?pessoa ?titulo ?prazo WHERE {
  ?emp a ex:Emprestimo ; ex:emprestadoPara ?p ; ex:itemEmprestado ?livro ;
       ex:dataPrevista ?prazo ; ex:ativo true .
  ?p ex:nome ?pessoa . ?livro ex:titulo ?titulo .
  FILTER(?prazo < "2026-11-03"^^xsd:date)
} ORDER BY ?prazo
''', 2),  # Esperado: emprestimo04 (28/10) e emprestimo02 (30/10).
    ('S7', 'Livros cuja estante habitual fica na biblioteca central', '''
SELECT ?titulo WHERE {
  ?livro ex:titulo ?titulo ; ex:guardadoEm/ex:pertenceBiblioteca ex:central .
} ORDER BY ?titulo
''', 8),  # Esperado: todos exceto livro04 e livro07. Caminho explícito, sem OWL.
    ('S8', 'Exemplares sem empréstimo ativo registrado', '''
SELECT ?livro ?titulo WHERE {
  ?livro ex:titulo ?titulo .
  FILTER NOT EXISTS { ?emp ex:itemEmprestado ?livro ; ex:ativo true . }
} ORDER BY ?livro
''', 5),  # Esperado: livro02, 04, 06, 09, 10. Ausência no cadastro, não verdade OWL.
]


def parte2(g):
    for numero, proposito, consulta, esperado in LEITURAS:
        print(f'\n[{numero}] {proposito}')
        resultado = g.query(PREFIXOS + consulta)
        if resultado.type == 'ASK':
            print('Resposta:', bool(resultado.askAnswer))
            assert bool(resultado.askAnswer) == esperado
        elif resultado.type == 'CONSTRUCT':
            mostrar_triplas(resultado.graph, resultado.graph)
            assert len(resultado.graph) == esperado
            assert not list(g.triples((None, EX.estaCom, None)))
        else:
            linhas = list(resultado)
            print(' | '.join(str(v) for v in resultado.vars))
            for linha in linhas:
                print(' | '.join(curto(g, valor) for valor in linha))
            print('Total:', len(linhas), 'linhas')
            assert len(linhas) == esperado, numero
            if numero == 'S2':
                assert [int(linha.paginas) for linha in linhas] == [48, 64, 80, 120]
            if numero == 'S3':
                assert sorted(int(linha.quantidade) for linha in linhas) == [2, 2, 3, 3]
            if numero == 'S6':
                assert [linha.emp for linha in linhas] == [EX.emprestimo04, EX.emprestimo02]
            if numero == 'S8':
                assert {linha.livro for linha in linhas} == {EX.livro02,EX.livro04,EX.livro06,EX.livro09,EX.livro10}


parte2(g)


[S1] Catálogo com autores, ordenado por título
titulo | autor
"A Casa do Lago"^^xsd:string | "Helena Martins Reis"^^xsd:string
"A Estrela Curiosa"^^xsd:string | "Helena Martins Reis"^^xsd:string
"A Viagem da Semente"^^xsd:string | "Clara Nunes de Souza"^^xsd:string
"Cartas do Amanhã"^^xsd:string | "Miguel Torres Lima"^^xsd:string
"Ciência na Escola"^^xsd:string | "Paulo Azevedo Dias"^^xsd:string
"História em Mapas"^^xsd:string | "Paulo Azevedo Dias"^^xsd:string
"Matemática em Passos"^^xsd:string | "Paulo Azevedo Dias"^^xsd:string
"O Caminho da Serra"^^xsd:string | "Clara Nunes de Souza"^^xsd:string
"O Jardim das Cores"^^xsd:string | "Miguel Torres Lima"^^xsd:string
"O Mistério da Ponte"^^xsd:string | "Clara Nunes de Souza"^^xsd:string
Total: 10 linhas

[S2] Leituras curtas: até 120 páginas
titulo | paginas
"A Estrela Curiosa"^^xsd:string | "48"^^xsd:integer
"O Jardim das Cores"^^xsd:string | "64"^^xsd:integer
"A Viagem da Semente"^^xsd:string | "80"^^xsd:integer
"Matemática em Passos"

## 10. Três formas de SPARQL Update
As atualizações são feitas numa cópia da mesma base, em sequência:

1. **S9 — INSERT DATA:** insere 7 triplas de um empréstimo temporário de livro10 para Elisa. Antes: registro ausente. Depois: presente.
2. **S10 — DELETE WHERE:** remove as 7 triplas desse registro. Antes: 7. Depois: 0, recuperando o estado inicial.
3. **S11 — DELETE/INSERT WHERE:** renova o prazo de Ana de 17/11 para 24/11/2026. Remove o antigo e insere o novo; não deixa dois prazos.

Os resultados são impressos antes/depois e verificados. O arquivo original não é sobrescrito. Para repetir a demonstração, execute a célula de novo: ela cria outra cópia limpa.

In [ ]:
def demonstrar_updates(original):
    """S9–S11: todos na mesma cópia da base original, sem salvar sobre o .ttl."""
    g = copiar(original)
    antes = set(original)
    print('\n[S9] INSERT DATA: registrar empréstimo temporário do livro10 para Elisa.')
    print('ANTES: triplas do emprestimo06 =', len(list(g.triples((EX.emprestimo06, None, None)))))
    g.update(PREFIXOS + '''INSERT DATA {
      ex:emprestimo06 a ex:Emprestimo ; ex:emprestadoPara ex:elisa ;
          ex:itemEmprestado ex:livro10 ; ex:registradoEm ex:central ;
          ex:dataEmprestimo "2026-11-03"^^xsd:date ;
          ex:dataPrevista "2026-11-17"^^xsd:date ; ex:ativo true .
    }''')
    # Esperado: 7 novas triplas; ASK muda de falso (registro ausente) para verdadeiro.
    mostrar_triplas(g, g.triples((EX.emprestimo06, None, None)))
    existe = bool(g.query(PREFIXOS + 'ASK { ex:emprestimo06 ex:itemEmprestado ex:livro10 }').askAnswer)
    print('DEPOIS: registro existe?', existe, '| Triplas adicionadas:', len(g)-len(original))
    assert existe and len(g) == len(original)+7

    print('\n[S10] DELETE WHERE: apagar o registro temporário criado em S9.')
    print('ANTES: triplas do emprestimo06 =', len(list(g.triples((EX.emprestimo06, None, None)))))
    g.update(PREFIXOS + 'DELETE WHERE { ex:emprestimo06 ?p ?o . }')
    # Esperado: zero triplas do registro e grafo igual ao original.
    print('DEPOIS: triplas do emprestimo06 =', len(list(g.triples((EX.emprestimo06, None, None)))))
    assert set(g) == antes

    print('\n[S11] DELETE/INSERT: renovar o prazo do empréstimo de Ana.')
    print('ANTES:', g.value(EX.emprestimo01, EX.dataPrevista))
    g.update(PREFIXOS + '''
DELETE { ex:emprestimo01 ex:dataPrevista ?antiga . }
INSERT { ex:emprestimo01 ex:dataPrevista "2026-11-24"^^xsd:date . }
WHERE  { ex:emprestimo01 ex:dataPrevista ?antiga ; ex:ativo true . }
''')
    # Esperado: remover 17/11 e inserir 24/11, sem deixar dois prazos.
    print('DEPOIS:', g.value(EX.emprestimo01, EX.dataPrevista))
    assert list(g.objects(EX.emprestimo01, EX.dataPrevista)) == [Literal('2026-11-24', datatype=XSD.date)]
    assert (EX.emprestimo01, EX.dataPrevista, Literal('2026-11-17', datatype=XSD.date)) not in g
    assert set(original) == antes
    print('Base original preservada; os updates ocorreram apenas em memória.')
    return g


g_atualizado = demonstrar_updates(g)


[S9] INSERT DATA: registrar empréstimo temporário do livro10 para Elisa.
ANTES: triplas do emprestimo06 = 0
ex:emprestimo06 | ex:ativo | "true"^^xsd:boolean
ex:emprestimo06 | ex:dataEmprestimo | "2026-11-03"^^xsd:date
ex:emprestimo06 | ex:dataPrevista | "2026-11-17"^^xsd:date
ex:emprestimo06 | ex:emprestadoPara | ex:elisa
ex:emprestimo06 | ex:itemEmprestado | ex:livro10
ex:emprestimo06 | ex:registradoEm | ex:central
ex:emprestimo06 | rdf:type | ex:Emprestimo
Total: 7 triplas
DEPOIS: registro existe? True | Triplas adicionadas: 7

[S10] DELETE WHERE: apagar o registro temporário criado em S9.
ANTES: triplas do emprestimo06 = 7
DEPOIS: triplas do emprestimo06 = 0

[S11] DELETE/INSERT: renovar o prazo do empréstimo de Ana.
ANTES: 2026-11-17
DEPOIS: 2026-11-24
Base original preservada; os updates ocorreram apenas em memória.


## 11. Demonstração adicional: o que OWL permite inferir?
Agora usamos outra cópia e aplicamos `DeductiveClosure(OWLRL_Semantics).expand()`. As consultas obrigatórias anteriores foram executadas na base explícita, sem esse enriquecimento.

| Fato não escrito diretamente | Por que pode ser inferido? |
|---|---|
| livro01 é Livro | Romance é subclasse de Livro |
| autor1 autorDe livro01 | Relação inversa de escritoPor |
| livro01 localizadoEm central | Subpropriedades e transitividade pela estante |
| anexo parceiraDe central | Simetria da parceria |

O código mostra `False` antes e `True` depois. Isso quer dizer ausência/presença da tripla no grafo, não que ela era logicamente falsa antes. A disjunção e a funcionalidade também são explicadas, com verificações de qualidade. O raciocinador é OWL-RL, não um validador completo de qualquer ontologia OWL.

In [ ]:
def demonstrar_owl(original):
    """Demonstra inferências OWL-RL numa cópia independente, sem updates."""
    inferido = copiar(original)
    exemplos = [
        ('I1: hierarquia de classes', (EX.livro01, RDF.type, EX.Livro)),
        ('I2: inverseOf', (EX.autor1, EX.autorDe, EX.livro01)),
        ('I3: subpropriedades + transitividade', (EX.livro01, EX.localizadoEm, EX.central)),
        ('I4: simetria da parceria', (EX.anexo, EX.parceiraDe, EX.central)),
    ]
    for _, tripla in exemplos:
        assert tripla not in original
    DeductiveClosure(OWLRL_Semantics).expand(inferido)
    for nome, tripla in exemplos:
        print(f'\n[{nome}] ANTES: {tripla in original} | DEPOIS: {tripla in inferido}')
        mostrar_triplas(inferido, [tripla])
        assert tripla in inferido
    # Disjunção: nenhum indivíduo desta base pode ser simultaneamente Livro e Pessoa/Local.
    for classe in [EX.Pessoa, EX.Local]:
        assert not (set(inferido.subjects(RDF.type, EX.Livro)) & set(inferido.subjects(RDF.type, classe)))
    print('\nFuncionalidade: cada registro tem um único item; cada exemplar tem um código.')
    print('Disjunção: não há Livro simultaneamente Pessoa ou Local nesta base.')
    print('OWL não é um formulário validador: domain/range inferem tipos; funcionalidade não é chave única.')
    return inferido


g_inferido = demonstrar_owl(g)


[I1: hierarquia de classes] ANTES: False | DEPOIS: True
ex:livro01 | rdf:type | ex:Livro
Total: 1 triplas

[I2: inverseOf] ANTES: False | DEPOIS: True
ex:autor1 | ex:autorDe | ex:livro01
Total: 1 triplas

[I3: subpropriedades + transitividade] ANTES: False | DEPOIS: True
ex:livro01 | ex:localizadoEm | ex:central
Total: 1 triplas

[I4: simetria da parceria] ANTES: False | DEPOIS: True
ex:anexo | ex:parceiraDe | ex:central
Total: 1 triplas

Funcionalidade: cada registro tem um único item; cada exemplar tem um código.
Disjunção: não há Livro simultaneamente Pessoa ou Local nesta base.
OWL não é um formulário validador: domain/range inferem tipos; funcionalidade não é chave única.


## 12. Sua consulta: altere o limite de páginas
Esta é uma consulta extra, além das 11 operações SPARQL exigidas e demonstradas. Altere apenas `limite_paginas`. O número é passado como literal por `initBindings`, sem montar uma consulta com texto digitado pelo usuário.

In [ ]:
limite_paginas = 100
if isinstance(limite_paginas, bool) or not isinstance(limite_paginas, int) or limite_paginas < 1:
    raise ValueError("Informe um número inteiro positivo de páginas.")
consulta = PREFIXOS + """
SELECT ?titulo ?paginas WHERE {
  ?livro ex:titulo ?titulo ; ex:paginas ?paginas .
  FILTER(?paginas <= ?limite)
} ORDER BY ?paginas
"""
for linha in g.query(consulta, initBindings={"limite": Literal(limite_paginas, datatype=XSD.integer)}):
    print(f"{linha.titulo}: {linha.paginas} páginas")

A Estrela Curiosa: 48 páginas
O Jardim das Cores: 64 páginas
A Viagem da Semente: 80 páginas
